# Build widget-only HTMLs

For each variant — the unbiased clustering across **all** digits, plus one per-digit clustergram — runs a single-cell notebook that builds the linked deck.gl + Celldega Clustergram pair, then exports to HTML with code cells hidden via `HTMLExporter(exclude_input=True)`.

We **don't** use `embed_minimal_html` here because its requireJS-based loader doesn't bundle anywidget's `_esm` module — the resulting HTML loads correctly in JupyterLab but errors out when opened standalone in a browser. nbconvert embeds anywidget properly, so we lean on that and just hide the code.

In [ ]:
import time
from pathlib import Path

import nbformat
from nbconvert import HTMLExporter
from nbconvert.preprocessors import ExecutePreprocessor

ROOT = Path.cwd()
TIMEOUT_S = 1800

VARIANTS = [
    {'label': 'All',   'mode': 'all',   'digit': None, 'n_clusters': 100, 'n_top_pixels': 500, 'n_examples': 16},
    {'label': 'Zero',  'mode': 'digit', 'digit': 0,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'One',   'mode': 'digit', 'digit': 1,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Two',   'mode': 'digit', 'digit': 2,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Three', 'mode': 'digit', 'digit': 3,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Four',  'mode': 'digit', 'digit': 4,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Five',  'mode': 'digit', 'digit': 5,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Six',   'mode': 'digit', 'digit': 6,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Seven', 'mode': 'digit', 'digit': 7,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Eight', 'mode': 'digit', 'digit': 8,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
    {'label': 'Nine',  'mode': 'digit', 'digit': 9,    'n_clusters':  10, 'n_top_pixels': 400, 'n_examples': 12},
]
print('Will build:', [v['label'] for v in VARIANTS])

## Build each widget-only HTML

In [ ]:
CELL_TEMPLATE = '''\
from mnist_celldega import silence_warnings
silence_warnings()

from ipywidgets import HBox, Layout
from mnist_celldega import (
    get_mnist_data, cluster_mnist,
    make_mnist_clustergram, make_mnist_viewer_widget, link_viewer_to_clustergram,
)

ds = get_mnist_data()
clusters = cluster_mnist(ds, mode={mode!r}, digit={digit!r}, n_clusters={n_clusters}, n_top_pixels={n_top_pixels}, n_examples={n_examples})
_, cgm = make_mnist_clustergram(clusters)
viewer = make_mnist_viewer_widget(clusters, height=720)
link_viewer_to_clustergram(viewer, cgm)
viewer.layout = Layout(width="620px", height="720px")
cgm.layout    = Layout(width="760px", height="720px")
HBox([viewer, cgm])
'''

def build_one(label, mode, digit, n_clusters, n_top_pixels, n_examples):
    out = ROOT / f'{label}_widget.html'
    print(f'\n=== {label} ===', flush=True)
    t0 = time.time()

    nb = nbformat.v4.new_notebook()
    nb.cells = [nbformat.v4.new_code_cell(
        CELL_TEMPLATE.format(mode=mode, digit=digit, n_clusters=n_clusters, n_top_pixels=n_top_pixels, n_examples=n_examples)
    )]
    nb.metadata = {
        'kernelspec': {'display_name': 'Python 3 (ipykernel)',
                       'language': 'python', 'name': 'python3'},
        'language_info': {'name': 'python'},
    }

    print('  executing...', flush=True)
    ep = ExecutePreprocessor(timeout=TIMEOUT_S, kernel_name='python3')
    ep.preprocess(nb, {'metadata': {'path': str(ROOT)}})

    exporter = HTMLExporter()
    exporter.exclude_input = True
    exporter.exclude_input_prompt = True
    exporter.exclude_output_prompt = True
    exporter.embed_images = True
    body, _ = exporter.from_notebook_node(nb)
    out.write_text(body, encoding='utf-8')

    size_mb = out.stat().st_size / 1e6
    dt = time.time() - t0
    print(f'  -> {out.name} ({size_mb:.1f} MB) in {dt:.1f}s', flush=True)
    return out

results = {}
for spec in VARIANTS:
    try:
        results[spec['label']] = ('ok', build_one(**spec))
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {e}', flush=True)
        results[spec['label']] = ('error', e)

## Summary

In [ ]:
for label, (status, info) in results.items():
    if status == 'ok':
        print(f'  OK   {label:<6} -> {info.name}')
    else:
        print(f'  FAIL {label:<6} -> {type(info).__name__}: {info}')